## Dask Delayed for general-purpose Python

This notebook shows how to use **Dask Delayed** to turn ordinary Python functions into a lazy task graph, then run that graph in parallel.

No xarray / Dask Array — just normal functions, `@delayed`, and `compute`.

**When to use Delayed:** custom Python pipelines you want to **describe first, run later** (static graphs, dependencies, fan-out / fan-in).

**Delayed vs Futures:** Delayed builds a lazy graph; Futures submit work immediately. Both target non-Dask-aware Python.

In [ ]:
import time
from dask import delayed, compute, visualize
from dask.distributed import Client, LocalCluster

### 1. Start a local cluster

Delayed also works with the default single-machine scheduler, but a `Client` lets you watch the dashboard.

In [ ]:
cluster = LocalCluster(n_workers=4, threads_per_worker=1)
client = Client(cluster)

print(client.dashboard_link)
client

### 2. Plain Python functions

Nothing Dask-specific here — stand-ins for slow custom work (parsing, IO, non-Dask libraries).

In [ ]:
def process_item(item: int, scale: float = 1.0) -> dict:
    time.sleep(2.0)
    return {"item": item, "value": (item ** 2) * scale}


def extract_value(result: dict) -> float:
    time.sleep(0.5)
    return result["value"]


def combine(values: list[float]) -> float:
    time.sleep(1.0)
    return sum(values)

### 3. Wrap with `@delayed`

Calling a delayed function does **not** run it. It returns a lazy `Delayed` object — a node in a task graph.

In [ ]:
@delayed
def process_item_delayed(item: int, scale: float = 1.0) -> dict:
    return process_item(item, scale)


@delayed
def extract_value_delayed(result: dict) -> float:
    return extract_value(result)


@delayed
def combine_delayed(values: list[float]) -> float:
    return combine(values)

In [ ]:
# Lazy: builds a task, does not execute yet
lazy_one = process_item_delayed(7, scale=2.0)
lazy_one

In [ ]:
# Trigger execution (uses the Client if one is present)
lazy_one.compute()

### 4. Build a parallel graph, then compute once

Create many delayed calls first; run them together with `compute` so Dask can schedule independent work in parallel.

In [ ]:
items = list(range(12))

lazy_results = [process_item_delayed(i, scale=1.5) for i in items]
lazy_results[:3]

In [ ]:
%%time
# One compute over the whole collection
results = compute(*lazy_results)
results

On a distributed cluster you can also do:

```python
futures = client.compute(lazy_results)
results = client.gather(futures)
```

That turns Delayed objects into Futures, then gathers concrete values.

In [ ]:
%%time
futures = client.compute(lazy_results)
results = client.gather(futures)
results[:3]

### 5. Dependencies (fan-out / fan-in)

Pass Delayed outputs into later delayed calls. Dask tracks dependencies and runs ready tasks as soon as inputs are available.

In [ ]:
lazy_items = [process_item_delayed(i) for i in range(6)]
lazy_values = [extract_value_delayed(r) for r in lazy_items]
lazy_total = combine_delayed(lazy_values)

lazy_total

In [ ]:
%%time
lazy_total.compute()

### 6. Inspect the graph (optional)

Useful when teaching or debugging dependencies. Requires Graphviz for image rendering; if unavailable, skip this cell.

In [ ]:
try:
    visualize(lazy_total, filename="delayed_graph", format="png", collapse_outputs=True)
except Exception as exc:
    print("Could not render graph:", exc)
    print(lazy_total.dask)

### 7. Alternative: `delayed(func)(...)` without a decorator

Same idea — wrap an existing function at the call site.

In [ ]:
lazy = [delayed(process_item)(i, scale=2.0) for i in range(4)]
compute(*lazy)

### 8. Batching large graphs (optional)

For very large fan-out, build/compute in batches so the graph stays manageable.

In [ ]:
def chunks(lst, n=5):
    return [lst[i:i + n] for i in range(0, len(lst), n)]


all_items = list(range(20))
all_results = []
print(f"All items: {all_items}")

for batch in chunks(all_items, 5):
    print(f"Batch: {batch}")
    lazy_batch = [process_item_delayed(i) for i in batch]
    all_results.extend(compute(*lazy_batch))

len(all_results), all_results[:3]

### 9. Clean up

In [ ]:
client.close()
cluster.close()

### Takeaways

1. Delayed parallelises **ordinary Python functions** by building a lazy task graph.
2. Calling a delayed function only builds a node; **`compute`** (or `client.compute` + `gather`) runs it.
3. Pass Delayed outputs into later delayed calls to express dependencies.
4. Prefer Delayed for **static “describe then run”** pipelines; prefer Futures for **imperative / dynamic** submit-and-react workflows.
5. For pure array math, use Dask Array / xarray instead of wrapping everything in Delayed.